In [1]:
import pandas as pd

In [17]:
def oracle(df, opt):
    print('Energy overhead of the dynamic approach compared to an oracle-based approach')

    mt = df[df['threads'] == 'mt'].copy()
    df = df[(df['threads'] != 'mt') & (df['threads'] != 'rt')].copy()

    mt['runtime'] = mt.apply(lambda row: row['runtime'] / df[(df['size'] == row['size']) & (df['threads'] == opt[row['size']])].iloc[0]['runtime'] * 100 - 100, axis=1)
    mt['energy']  = mt.apply(lambda row: row['energy']  / df[(df['size'] == row['size']) & (df['threads'] == opt[row['size']])].iloc[0]['energy']  * 100 - 100, axis=1)

    return mt.groupby('size').mean(numeric_only=True).round(0).astype(int)

In [18]:
def runtime(df):
    print('Energy overhead of the dynamic approach compared to an runtime-based approach')

    mt = df[df['threads'] == 'mt'].copy()
    rt = df[df['threads'] == 'rt'].copy()

    mt['runtime'] = mt.apply(lambda row: row['runtime'] / rt[rt['size'] == row['size']].iloc[0]['runtime'] * 100 - 100, axis=1)
    mt['energy']  = mt.apply(lambda row: row['energy']  / rt[rt['size'] == row['size']].iloc[0]['energy']  * 100 - 100, axis=1)

    return mt.groupby('size').mean(numeric_only=True).round(0).astype(int)

In [19]:
def mean(df):
    print('\nEnergy overhead of the dynamic approach compared to a static approach')
    
    mt = df[df['threads'] == 'mt'].copy()
    df = df[(df['threads'] != 'mt') & (df['threads'] != 'rt')].copy()

    # Normalise runtime and energy consumption
    df['runtime'] = df.apply(lambda row: 100 - mt[mt['size'] == row['size']].iloc[0]['runtime'] / row['runtime'] * 100, axis=1)
    df['energy']  = df.apply(lambda row: 100 - mt[mt['size'] == row['size']].iloc[0]['energy']  / row['energy']  * 100, axis=1)

    # Drop unused columns and rows
    df = df.drop('size', axis=1)
    df = df[df['threads'] != 'mt']
    df['threads'] = df['threads'].astype(int)

    # Mean per thread-count
    return df.groupby('threads').mean(numeric_only=True).round(0).astype(int)

In [20]:
df = pd.read_csv('data/compare_matmul.csv')
df = df.drop(['runtimesd', 'energysd'], axis=1)
print(oracle(df, { 500: '16', 1000: '16', 1500: '1' }))
print(mean(df))

Energy overhead of the dynamic approach compared to an oracle-based approach
      runtime  energy
size                 
500        -1      -3
1000       10       1
1500      -24      76

Energy overhead of the dynamic approach compared to a static approach
         runtime  energy
threads                 
1             55      -6
8             44      18
12            17       5
16            -5       0


In [27]:
df = pd.read_csv('data/compare_nbody.csv')
df = df.drop(['runtimesd', 'energysd'], axis=1)
print(oracle(df, { 10000: '16', 25000: '16', 40000: '16' }))
print(mean(df))

Energy overhead of the dynamic approach compared to an oracle-based approach
       runtime  energy
size                  
10000        6       3
25000        1      -2
40000      -68     -68

Energy overhead of the dynamic approach compared to a static approach
         runtime  energy
threads                 
8             57      32
12            36      24
16            20      22


In [22]:
df = pd.read_csv('data/compare_relax.csv')
df = df.drop(['runtimesd', 'energysd'], axis=1)
print(oracle(df, { 10000: '16', 25000: '14', 40000: '12' }))
print(mean(df))

Energy overhead of the dynamic approach compared to an oracle-based approach
       runtime  energy
size                  
10000        5       7
25000        4       4
40000       34      16

Energy overhead of the dynamic approach compared to a static approach
         runtime  energy
threads                 
8             32      10
12            -2      -5
14            -8      -2
16            -7       4


In [25]:
df = pd.read_csv('data/compare_rust.csv')
df = df.drop(['runtimesd', 'energysd'], axis=1)

y = df[df['pin'] == True].copy()
print(oracle(y, { 500: '16', 1000: '16', 1500: '12' }))
print(runtime(y))
print(mean(y))

Energy overhead of the dynamic approach compared to an oracle-based approach
      pin  runtime  energy
size                      
500     1        5       2
1000    1        7       4
1500    1       12       8
Energy overhead of the dynamic approach compared to an runtime-based approach
      pin  runtime  energy
size                      
500     1       -6      -3
1000    1       -3      -1
1500    1       -7      -3

Energy overhead of the dynamic approach compared to a static approach
         pin  runtime  energy
threads                      
1          1       87      66
8          1       49      23
12         1       -4      -3
16         1       -1       2


In [26]:
df = pd.read_csv('data/compare_rust.csv')
df = df.drop(['runtimesd', 'energysd'], axis=1)

n = df[df['pin'] == False].copy()
print(oracle(n, { 500: '16', 1000: '16', 1500: '8' }))
print(runtime(n))
print(mean(n))

Energy overhead of the dynamic approach compared to an oracle-based approach
      pin  runtime  energy
size                      
500     0        2       1
1000    0        2       2
1500    0       22      15
Energy overhead of the dynamic approach compared to an runtime-based approach
      pin  runtime  energy
size                      
500     0       -3      -2
1000    0        0       0
1500    0       -1       2

Energy overhead of the dynamic approach compared to a static approach
         pin  runtime  energy
threads                      
1          0       87      67
8          0        3      -1
12         0        1       1
16         0        3       5
